# Using a decision tree to predict CTR

In [1]:
import os
import pandas as pd
import time
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import roc_auc_score
import numpy as np

In [2]:
# From https://www.kaggle.com/competitions/avazu-ctr-prediction/data

In [3]:
n_rows = 300000
df = pd.read_csv("train.gz", nrows=n_rows)
print(df.head(5))

             id  click      hour    C1  banner_pos   site_id site_domain  \
0  1.000009e+18      0  14102100  1005           0  1fbe01fe    f3845767   
1  1.000017e+19      0  14102100  1005           0  1fbe01fe    f3845767   
2  1.000037e+19      0  14102100  1005           0  1fbe01fe    f3845767   
3  1.000064e+19      0  14102100  1005           0  1fbe01fe    f3845767   
4  1.000068e+19      0  14102100  1005           1  fe8cc448    9166c161   

  site_category    app_id app_domain  ... device_type device_conn_type    C14  \
0      28905ebd  ecad2386   7801e8d9  ...           1                2  15706   
1      28905ebd  ecad2386   7801e8d9  ...           1                0  15704   
2      28905ebd  ecad2386   7801e8d9  ...           1                0  15704   
3      28905ebd  ecad2386   7801e8d9  ...           1                0  15706   
4      0569f928  ecad2386   7801e8d9  ...           1                0  18993   

   C15  C16   C17  C18  C19     C20  C21  
0  320   50  

In [4]:
X = df.drop(['click', 'id', 'hour', 'device_id', 'device_ip'], axis=1).values
Y = df['click'].values

In [5]:
print(X.shape)

(300000, 19)


In [6]:
n_train = int(n_rows * 0.9)
X_train = X[:n_train]
Y_train = Y[:n_train]
X_test = X[n_train:]
Y_test = Y[n_train:]

In [7]:
enc = OneHotEncoder(handle_unknown='ignore')
X_train_enc = enc.fit_transform(X_train)

In [8]:
X_train_enc[0]
print(X_train_enc[0])

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 19 stored elements and shape (1, 8204)>
  Coords	Values
  (0, 2)	1.0
  (0, 6)	1.0
  (0, 188)	1.0
  (0, 2608)	1.0
  (0, 2679)	1.0
  (0, 3771)	1.0
  (0, 3885)	1.0
  (0, 3929)	1.0
  (0, 4879)	1.0
  (0, 7315)	1.0
  (0, 7319)	1.0
  (0, 7475)	1.0
  (0, 7824)	1.0
  (0, 7828)	1.0
  (0, 7869)	1.0
  (0, 7977)	1.0
  (0, 7982)	1.0
  (0, 8021)	1.0
  (0, 8189)	1.0


In [9]:
X_test_enc = enc.transform(X_test)

In [10]:
parameters = {'max_depth': [3, 10, None]}
decision_tree = DecisionTreeClassifier(criterion='gini', min_samples_split=30)
grid_search = GridSearchCV(decision_tree, parameters, n_jobs=-1, cv=3, scoring='roc_auc')

In [11]:
start = time.perf_counter()
grid_search.fit(X_train_enc, Y_train)
print(f"Training time: {time.perf_counter() - start:.4f}s")

Training time: 20.2472s


In [12]:
print(grid_search.best_params_)

{'max_depth': 10}


In [13]:
decision_tree_best = grid_search.best_estimator_
pos_prob = decision_tree_best.predict_proba(X_test_enc)[:, 1]

In [14]:
print(f'[Decision Tree]The ROC AUC on testing set is: {roc_auc_score(Y_test, pos_prob):.3f}')

[Decision Tree]The ROC AUC on testing set is: 0.719


In [15]:
len(X_train)

270000

In [16]:
pos_prob = np.zeros(len(Y_test))
click_index = np.random.choice(len(Y_test), int(len(Y_test) *  Y_train.sum()/(len(X_train)+len(Y_train))), replace=False)
pos_prob[click_index] = 1

In [17]:
print(f'[Base classifier] The ROC AUC on testing set is: {roc_auc_score(Y_test, pos_prob):.3f}')

[Base classifier] The ROC AUC on testing set is: 0.501


## Random forest

In [18]:
from sklearn.ensemble import RandomForestClassifier

In [19]:
random_forest = RandomForestClassifier(n_estimators=100, criterion='gini', min_samples_split=30, n_jobs=-1)
grid_search = GridSearchCV(random_forest, parameters, n_jobs=-1, cv=3, scoring='roc_auc')

In [20]:
start = time.perf_counter()
grid_search.fit(X_train_enc, Y_train)
print(f"Training time: {time.perf_counter() - start:.4f}s")

Training time: 180.7271s


In [21]:
print(grid_search.best_params_)
print(grid_search.best_score_)

{'max_depth': None}
0.7357301310162024


In [22]:
random_forest_best = grid_search.best_estimator_
pos_prob = random_forest_best.predict_proba(X_test_enc)[:, 1]

In [23]:
print(f'[Random Forest] The ROC AUC on testing set is: {roc_auc_score(Y_test, pos_prob):.3f}')

[Random Forest] The ROC AUC on testing set is: 0.758


## Gradient boosted trees

In [24]:
import xgboost as xgb

In [25]:
model = xgb.XGBClassifier(learning_rate=0.1, max_depth=10, n_estimators=1000)

In [26]:
start = time.perf_counter()
model.fit(X_train_enc, Y_train)
print(f"Training time: {time.perf_counter() - start:.4f}s")

Training time: 10.2422s


In [27]:
print(f'[xGB]The ROC AUC on testing set is: {roc_auc_score(Y_test, pos_prob):.3f}')

[xGB]The ROC AUC on testing set is: 0.758
